In [ ]:
@staticmethod
def verify_analyzer_long(analyzer, n_tickers: int = 5) -> TaskResult:  # Add n_tickers
    """
    FULL SPECTRUM AUDIT: Independent recalculation of ATRP, Survival, and Strategy Ranking.
    1. Performance (3 Periods, Warm-Start ATRP, Decimal Mode)
    2. Survival (Liquidity/Quality Gate)
    3. Universal Selection (Strategy Math reconciliation for ALL candidates)
    """
    # --- LATE IMPORTS to avoid circular dependency ---
    from strategy.registry import METRIC_REGISTRY

    try:
        from IPython.display import display
    except ImportError:
        display = print

    res = getattr(analyzer, "last_run", None)
    engine = getattr(analyzer, "engine", None)
    if not res or not res.debug_data:
        print("❌ Audit Aborted: No debug data.")
        return

    debug = res.debug_data
    inputs = debug["inputs_snapshot"]
    m = res.perf_metrics

    print("\n" + "=" * 85)
    print(f"🛡️  STARTING NUCLEAR AUDIT | {res.decision_date.date()} | {inputs.metric}")
    print("*" * 85)
    print(
        "⚠️  ASSUMPTION: Verification logic is independent, but trusts Engine source DataFrames"
    )
    print(
        "    (engine.features_df, engine.df_close, and debug['portfolio_raw_components'])"
    )
    print("*" * 85)
    print("=" * 85)

    periods = {
        "Full": (res.start_date, res.holding_end_date),
        "Lookback": (res.start_date, res.decision_date),
        "Holding": (res.buy_date, res.holding_end_date),
    }

    # --------------------------------------------------------------------------
    # HELPER 1: MANUAL ATRP CALCULATION (DECIMAL MODE)
    # --------------------------------------------------------------------------
    def calculate_manual_atrp_warm(df_ohlcv, features_df, df_close_matrix, start_date):
        df = df_ohlcv.copy()

        available_tickers = df.index.get_level_values("Ticker").unique()
        if len(available_tickers) == 0:
            return pd.DataFrame(), pd.DataFrame()

        seed_atrp_all = features_df.xs(start_date, level="Date")["ATRP"]

        # Intersect to find valid debug candidate
        valid_debug_tickers = [t for t in available_tickers if t in seed_atrp_all.index]
        if not valid_debug_tickers:
            return pd.DataFrame(), pd.DataFrame()

        df["PC"] = df.groupby(level="Ticker")["Adj Close"].shift(1)

        # STRICT TR: skipna=False matches Engine logic
        tr = pd.concat(
            [
                df["Adj High"] - df["Adj Low"],
                (df["Adj High"] - df["PC"]).abs(),
                (df["Adj Low"] - df["PC"]).abs(),
            ],
            axis=1,
        ).max(axis=1, skipna=False)

        seed_price = df_close_matrix.loc[start_date]

        # DECIMAL MODE: No multiplication/division by 100
        # Formula: SeedATR = ATRP(Decimal) * Price
        seed_atr = seed_atrp_all.reindex(available_tickers) * seed_price.reindex(
            available_tickers
        )

        alpha = 1 / 14

        def ewm_warm(group):
            ticker = group.name
            initial_val = seed_atr.get(ticker, group.iloc[0])
            vals = group.values
            results = np.zeros_like(vals)
            results[0] = initial_val

            for i in range(1, len(vals)):
                results[i] = (vals[i] * alpha) + (results[i - 1] * (1 - alpha))
            return pd.Series(results, index=group.index)

        manual_atr = tr.groupby(level="Ticker", group_keys=False).apply(ewm_warm)
        prices_wide = df["Adj Close"].unstack(level=0)

        # DECIMAL MODE OUTPUT: ATR / Price
        manual_atrp_decimal = manual_atr.unstack(level=0) / prices_wide

        return (
            manual_atrp_decimal,
            tr.unstack(level=0) / prices_wide,
        )

    # --------------------------------------------------------------------------
    # HELPER 2: PERIOD AUDIT RUNNER
    # --------------------------------------------------------------------------
    def run_period_audit(df_p, df_atrp, df_trp, weights):
        if df_p.empty:
            return 0, 0, 0, 0

        norm = df_p.div(df_p.bfill().iloc[0])
        equity = (norm * weights).sum(axis=1)
        # Portfolio risk is rebalance everyday, portfolio is not rebalanced everyday
        drift_w = (norm * weights).div(equity, axis=0)

        # Weighted Volatility
        p_atrp_manual = (drift_w * df_atrp).sum(axis=1)
        p_trp_manual = (drift_w * df_trp).sum(axis=1)

        # FIX: The Engine uses Log Returns for Gain
        # np.log(Ending_Equity) assumes starting equity was 1.0
        gain = np.log(equity.iloc[-1])

        rets = equity.pct_change().dropna()
        if rets.empty:
            return 0, 0, 0, 0

        # Ensure annualization factor matches GLOBAL_SETTINGS (usually 252)
        sharpe = (rets.mean() / rets.std() * np.sqrt(252)) if rets.std() > 0 else 0
        return (
            gain,
            sharpe,
            rets.mean() / p_atrp_manual.loc[rets.index].mean(),
            rets.mean() / p_trp_manual.loc[rets.index].mean(),
        )

    # --------------------------------------------------------------------------
    # PART 1: PERFORMANCE RECONCILIATION
    # --------------------------------------------------------------------------
    audit_rows = []
    targets = [
        ("p", debug["portfolio_raw_components"], res.initial_weights, "Group"),
        (
            "b",
            debug["benchmark_raw_components"],
            pd.Series({inputs.benchmark_ticker: 1.0}),
            "Benchmark",
        ),
    ]

    for prefix, components, weights, entity_name in targets:
        m_atrp, m_trp = calculate_manual_atrp_warm(
            components["ohlcv_raw"],
            engine.features_df,
            engine.df_close,
            res.start_date,
        )
        m_price = components["prices"]

        for p_label, (d_start, d_end) in periods.items():
            mg, ms, msa, mst = run_period_audit(
                m_price.loc[d_start:d_end],
                m_atrp.loc[d_start:d_end],
                m_trp.loc[d_start:d_end],
                weights,
            )

            for m_name, m_val, e_key in [
                ("Gain", mg, f"{p_label.lower()}_{prefix}_gain"),
                ("Sharpe", ms, f"{p_label.lower()}_{prefix}_sharpe"),
                ("Sharpe (ATRP)", msa, f"{p_label.lower()}_{prefix}_sharpe_atrp"),
                ("Sharpe (TRP)", mst, f"{p_label.lower()}_{prefix}_sharpe_trp"),
            ]:
                e_val = m.get(e_key, 0)
                audit_rows.append(
                    {
                        "Entity": entity_name,
                        "Period": p_label,
                        "Metric": m_name,
                        "Engine": e_val,
                        "Manual": m_val,
                        "Delta": e_val - m_val,
                    }
                )

    df_perf = pd.DataFrame(audit_rows)
    df_perf["Status"] = df_perf["Delta"].apply(
        lambda x: "✅ PASS" if abs(x) < 1e-7 else "❌ FAIL"
    )
    print("📝 1. PERFORMANCE RECONCILIATION")
    display(
        df_perf.pivot_table(
            index=["Entity", "Metric"],
            columns="Period",
            values="Status",
            aggfunc="first",
        )
    )

    # --------------------------------------------------------------------------
    # PART 2: SURVIVAL AUDIT (Liquidity/Quality Gate)
    # --------------------------------------------------------------------------
    print("\n" + "=" * 85)
    print("📝 2. SURVIVAL AUDIT")
    if inputs.universe_subset:
        print(
            "   Mode: CASCADE/SUBSET | Logic: Quality filters bypassed per design. | ✅ BYPASS"
        )
    else:
        audit_liq = debug.get("audit_liquidity")

        # SAFETY CHECK: Handle missing or None audit_liquidity data
        if audit_liq is None:
            print("   ⚠️  WARNING: audit_liquidity data not found in debug output.")
            print(
                "   Status: ❌ SKIP (Cannot verify survival logic without debug data)"
            )
        else:
            snapshot = audit_liq["universe_snapshot"]
            thresholds = inputs.quality_thresholds

            m_cutoff = max(
                snapshot["RollMedDollarVol"].quantile(
                    thresholds["min_liquidity_percentile"]
                ),
                thresholds["min_median_dollar_volume"],
            )
            m_survivors = snapshot[
                (snapshot["RollMedDollarVol"] >= m_cutoff)
                & (snapshot["RollingStalePct"] <= thresholds["max_stale_pct"])
                & (snapshot["RollingSameVolCount"] <= thresholds["max_same_vol_count"])
            ]
            s_match = (
                "✅ PASS"
                if audit_liq["tickers_passed"] == len(m_survivors)
                else "❌ FAIL"
            )
            print(
                f"   Survival Integrity: {s_match} (Engine: {audit_liq['tickers_passed']} vs Auditor: {len(m_survivors)})"
            )

    # --- PART 3: UNIVERSAL SELECTION AUDIT ---
    if inputs.mode == "Ranking":
        print("\n" + "=" * 85)
        print(f"📝 3. UNIVERSAL SELECTION AUDIT | Strategy: {inputs.metric}")

        if "full_universe_ranking" not in debug:
            print("❌ Audit Error: 'full_universe_ranking' not found in debug data.")
            return

        # --- FIX: Define survivors from the Engine's own result list ---
        eng_rank_df = debug["full_universe_ranking"]
        survivors = eng_rank_df.index.tolist()

        # 1. Identify the Trading Calendar for this window
        # We get this from engine.df_close to ensure alignment with the Engine's reality
        all_dates = engine.df_close.index
        full_window = all_dates[
            (all_dates >= res.start_date) & (all_dates <= res.decision_date)
        ]

        # 2. Define the 'Active Window' (Skip the anchor/P0)
        # This is exactly what the Engine does: active_dates = full_window_dates[1:]
        active_dates = full_window[1:]

        # 3. Slice features using ONLY the active dates for the mean
        idx = pd.IndexSlice
        feat_period_active = engine.features_df.loc[idx[survivors, active_dates], :]

        # Calculation of means on the active window only
        atrp_lb_mean = feat_period_active["ATRP"].groupby(level="Ticker").mean()
        trp_lb_mean = feat_period_active["TRP"].groupby(level="Ticker").mean()

        # Pull the current snapshots
        feat_now = engine.features_df.xs(res.decision_date, level="Date").reindex(
            survivors
        )
        macro_now = engine.macro_df.loc[res.decision_date]

        # 4. Pull price data including the anchor (so pct_change has a starting point)
        lb_prices = engine.df_close.loc[full_window, survivors]

        # Rebuild Observation exactly like the Engine's _build_observation
        audit_obs = MarketObservation(
            lookback_close=lb_prices,
            lookback_returns=lb_prices.pct_change(),  # P0 is NaN, P1 is first valid return
            atrp=atrp_lb_mean,  # Mean of [P1...PN]
            trp=trp_lb_mean,  # Mean of [P1...PN]
            atr=feat_now.get("ATR"),
            rsi=feat_now["RSI"],
            consistency=feat_now["Consistency"],
            mom_21=feat_now["Mom_21"],
            ir_63=feat_now["IR_63"],
            beta_63=feat_now["Beta_63"],
            dd_21=feat_now["DD_21"],
            macro_trend=macro_now["Macro_Trend"],
            macro_trend_vel=macro_now.get(
                "Macro_Trend_Vel", 0.0
            ),  # Added missing field
            macro_vix_z=macro_now["Macro_Vix_Z"],
            macro_vix_ratio=macro_now["Macro_Vix_Ratio"],
        )
        # Run Manual Registry Math on Full Universe
        manual_scores = METRIC_REGISTRY[inputs.metric](audit_obs)

        # Compare
        audit_data = []
        for i, (ticker, row) in enumerate(eng_rank_df.iterrows()):
            eng_val = row["Strategy_Score"]
            man_val = manual_scores.get(ticker, np.nan)
            delta = eng_val - man_val

            status = "✅ PASS" if np.isclose(eng_val, man_val, atol=1e-8) else "❌ FAIL"

            audit_data.append(
                {
                    "Rank": i + 1,
                    "Ticker": ticker,
                    "Engine": eng_val,
                    "Manual": man_val,
                    "Delta": delta,
                    "Status": status,
                }
            )

        df_audit_all = pd.DataFrame(audit_data).set_index("Rank")
        n_pass = (df_audit_all["Status"] == "✅ PASS").sum()
        n_fail = len(df_audit_all) - n_pass

        print(f"   Scope: Evaluated {len(df_audit_all)} candidates (Full Universe).")
        print(f"   Result: {n_pass} PASSED | {n_fail} FAILED")

    if n_fail > 0:
        print(f"⚠️  DISPLAYING FAILURES (Top {n_tickers}):")
        display(df_audit_all[df_audit_all["Status"] == "❌ FAIL"].head(n_tickers))
    else:
        print(
            f"   All scores match registry math. {inputs.metric} results of the first {n_tickers} tickers"
        )
        display(
            df_audit_all.head(n_tickers).style.format(
                "{:.8f}", subset=["Engine", "Manual", "Delta"]
            )
        )

    print("=" * 85)

    # After Performance Part:
    perf_failed = (df_perf["Status"] == "❌ FAIL").any()

    # After Selection Part:
    # if n_fail > 0: ranking_failed = True

    if perf_failed or n_fail > 0:
        return TaskResult(ok=False, msg="Nuclear audit failed reconciliation.")

    return TaskResult(ok=True, msg="Nuclear audit passed all checks.")

In [ ]:
    "Sharpe (ATRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs.lookback_returns, obs.atrp
    ),